In [1]:
# ==============================================================================
# 1. PERSIAPAN DAN MUAT DATA
# ==============================================================================
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# Nama file CSV Anda
file_path = '/content/drive/MyDrive/komen positif negatif.csv'

# Muat data dari file CSV
try:
    df = pd.read_csv(file_path)
    print("✅ Data berhasil dimuat.")
    print("Jumlah data:", len(df))
    print("\n5 baris data pertama:")
    print(df.head())
except FileNotFoundError:
    print(f"❌ ERROR: File '{file_path}' tidak ditemukan. Pastikan file sudah diupload ke sesi Colab Anda.")
    exit()

# Kolom yang akan digunakan
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'Sentimen'

# Cek apakah kolom yang dibutuhkan ada
if TEXT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
    print(f"❌ ERROR: Kolom '{TEXT_COLUMN}' atau '{LABEL_COLUMN}' tidak ditemukan dalam file CSV.")
    exit()

✅ Data berhasil dimuat.
Jumlah data: 323

5 baris data pertama:
  Sentimen       author                                               text  \
0  Positif   User_Bijak  Penyampaian yang sangat berbobot dan cerdas. S...   
1  Positif    AkbarGlow  Setuju banget sama poin-poinnya. Akhirnya ada ...   
2  Positif  Rani_Cerdas  Diskusi yang sangat mencerahkan. Saya mendapat...   
3  Positif  Tulus_Pikir  Sangat menginspirasi. Indonesia butuh lebih ba...   
4  Positif  BintangJaya  Salut buat Om Deddy dan Ferry. Pertanyaan dan ...   

    publishedAt likeCount  
0  5 months ago      1.5K  
1  5 months ago       890  
2  5 months ago      1.2K  
3  4 months ago       550  
4  4 months ago      2.1K  


In [2]:
# ==============================================================================
# 2. PREPROCESSING TEKS
# ==============================================================================

# Definisikan fungsi untuk membersihkan teks (case folding, menghilangkan tanda baca, dll.)
def clean_text(text):
    # 1. Case Folding (mengubah ke huruf kecil)
    text = text.lower()

    # 2. Menghilangkan Tanda Baca (punctuation)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Menghilangkan Angka (numerik)
    text = re.sub(r'\d+', '', text)

    # 4. Menghilangkan spasi berlebihan dan memotong spasi di awal/akhir
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)

    return text

print("\n--- Memulai Preprocessing Teks ---")
# Terapkan fungsi cleaning ke kolom 'text'
df['text_clean'] = df[TEXT_COLUMN].apply(clean_text)

print("✅ Preprocessing selesai.")
print("Contoh teks sebelum vs sesudah:")
print(f"Sebelum: {df[TEXT_COLUMN].iloc[0]}")
print(f"Sesudah: {df['text_clean'].iloc[0]}")


--- Memulai Preprocessing Teks ---
✅ Preprocessing selesai.
Contoh teks sebelum vs sesudah:
Sebelum: Penyampaian yang sangat berbobot dan cerdas. Salut dengan argumen Bang Ferry.
Sesudah: penyampaian yang sangat berbobot dan cerdas salut dengan argumen bang ferry


In [3]:
# ==============================================================================
# 3. PEMBAGIAN DATA & EKSTRAKSI FITUR (TF-IDF)
# ==============================================================================

# Pisahkan fitur (X) dan label (y)
X = df['text_clean']
y = df[LABEL_COLUMN]

# Pisahkan data menjadi data training dan data testing (misalnya 80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\n--- Pembagian Data Training/Testing (20%) ---")
print(f"Data Training: {len(X_train)} sampel")
print(f"Data Testing: {len(X_test)} sampel")

# Inisialisasi TF-IDF Vectorizer
# max_features=500 untuk membatasi jumlah fitur (kata) yang paling sering muncul
tfidf_vectorizer = TfidfVectorizer(max_features=500)

# Lakukan proses fit hanya pada data training, kemudian transform pada training dan testing
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("\n--- Ekstraksi Fitur (TF-IDF) ---")
print(f"✅ Dimensi Matriks Fitur Training: {X_train_tfidf.shape}")
print(f"✅ Jumlah Fitur (Kata) yang diekstrak: {len(tfidf_vectorizer.get_feature_names_out())}")


--- Pembagian Data Training/Testing (20%) ---
Data Training: 258 sampel
Data Testing: 65 sampel

--- Ekstraksi Fitur (TF-IDF) ---
✅ Dimensi Matriks Fitur Training: (258, 500)
✅ Jumlah Fitur (Kata) yang diekstrak: 500


In [4]:
# ==============================================================================
# 4. KLASIFIKASI DENGAN NAIVE BAYES
# ==============================================================================

print("\n--- Pelatihan Model Naive Bayes ---")
# Inisialisasi model Naive Bayes (MultinomialNB cocok untuk fitur hitungan/frekuensi seperti TF-IDF)
nb_model = MultinomialNB()

# Latih model menggunakan data training yang sudah diekstrak fiturnya
nb_model.fit(X_train_tfidf, y_train)

print("✅ Pelatihan model selesai.")

# Prediksi pada data testing
y_pred = nb_model.predict(X_test_tfidf)


--- Pelatihan Model Naive Bayes ---
✅ Pelatihan model selesai.


In [5]:
# ==============================================================================
# 5. EVALUASI MODEL
# ==============================================================================

print("\n=======================================================")
print("                   HASIL EVALUASI MODEL                ")
print("=======================================================")

# Hitung Akurasi
accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi Model: {accuracy*100:.2f}%")
print("\nLaporan Klasifikasi (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_pred))


                   HASIL EVALUASI MODEL                
Akurasi Model: 87.69%

Laporan Klasifikasi (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

     Negatif       0.89      0.89      0.89        37
     Positif       0.86      0.86      0.86        28

    accuracy                           0.88        65
   macro avg       0.87      0.87      0.87        65
weighted avg       0.88      0.88      0.88        65



PERCOBAAN 1

In [6]:
# ==============================================================================
# 6. CONTOH PENGUJIAN KOMENTAR BARU
# ==============================================================================

def predict_sentiment(text_input, model, vectorizer):
    # 1. Preprocessing Teks Input
    clean_input = clean_text(text_input)
    # 2. Ekstraksi Fitur (menggunakan vectorizer yang sudah di-fit)
    vectorized_input = vectorizer.transform([clean_input])
    # 3. Prediksi
    prediction = model.predict(vectorized_input)[0]
    return prediction

print("\n--- Contoh Prediksi Komentar Baru ---")

# Komentar Positif
new_comment_positive = "Saya sangat suka dengan ketenangan dan wibawanya.."
predicted_label_pos = predict_sentiment(new_comment_positive, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_positive}' -> Prediksi: {predicted_label_pos}")

# Komentar Negatif
new_comment_negative = "Cuma cari panggung biar viral, isinya tidak ada solusi konkret."
predicted_label_neg = predict_sentiment(new_comment_negative, nb_model, tfidf_vectorizer)
print(f"Komentar: '{new_comment_negative}' -> Prediksi: {predicted_label_neg}")


--- Contoh Prediksi Komentar Baru ---
Komentar: 'Saya sangat suka dengan ketenangan dan wibawanya..' -> Prediksi: Positif
Komentar: 'Cuma cari panggung biar viral, isinya tidak ada solusi konkret.' -> Prediksi: Negatif


**2 BOW**

In [22]:
# --- Implementasi Fitur Count (Bag of Words) ---
from sklearn.feature_extraction.text import CountVectorizer

vektor_bow = CountVectorizer()
train_features_bow = vektor_bow.fit_transform(X_train)
test_features_bow = vektor_bow.transform(X_test)

print(f"{'='*30}\n EVALUASI MODEL: BAG OF WORDS \n{'='*30}")

# Definisikan dictionary 'models' yang berisi model yang akan dievaluasi
models = {
    "Multinomial Naive Bayes": MultinomialNB()
}

# Melakukan iterasi untuk setiap algoritma di dalam dictionary models
for label_model, algoritma in models.items():
    # Melatih algoritma dengan data training versi BoW
    algoritma.fit(train_features_bow, y_train)

    # Melakukan pengujian pada data test
    hasil_prediksi = algoritma.predict(test_features_bow)

    # Menampilkan laporan performa
    print(f"\nHasil Pengujian - Algoritma: {label_model}")
    print("-" * 25)
    print(classification_report(y_test, hasil_prediksi))


 EVALUASI MODEL: BAG OF WORDS 

Hasil Pengujian - Algoritma: Multinomial Naive Bayes
-------------------------
              precision    recall  f1-score   support

     Negatif       0.91      0.86      0.89        37
     Positif       0.83      0.89      0.86        28

    accuracy                           0.88        65
   macro avg       0.87      0.88      0.88        65
weighted avg       0.88      0.88      0.88        65



**3. Word2Vec**

In [27]:
!pip install gensim
import gensim
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# --- Tahap 1: Inisialisasi Model Word2Vec ---
# Tokenisasi data latih menjadi list kata
data_kata = [kalimat.split() for kalimat in X_train]

# Membangun model embedding Word2Vec
w2v_engine = gensim.models.Word2Vec(
    sentences=data_kata,
    vector_size=100,
    window=5,
    min_count=1
)

# --- Tahap 2: Fungsi Representasi Vektor ---
def get_sentence_embedding(teks):
    # Mengambil vektor untuk setiap kata yang tersedia di kamus w2v
    vektor_kata = [w2v_engine.wv[kata] for kata in teks.split() if kata in w2v_engine.wv]

    # Jika ada vektor yang ditemukan, hitung rata-ratanya; jika kosong, berikan array nol
    return np.mean(vektor_kata, axis=0) if vektor_kata else np.zeros(100)

# --- Tahap 3: Transformasi Data ---
# Mengubah kumpulan teks menjadi matriks fitur numerik
fitur_latih_w2v = np.array([get_sentence_embedding(txt) for txt in X_train])
fitur_uji_w2v = np.array([get_sentence_embedding(txt) for txt in X_test])

# Daftar algoritma yang akan diuji
koleksi_model = {
    "SVM_Classifier": SVC(),
    "Log_Regression": LogisticRegression(max_iter=2000)
}

# --- Tahap 4: Pelatihan dan Evaluasi ---
print(f"\n{'*' * 10} PERFORMA WORD2VEC {'*' * 10}\n")

for label, m_obj in koleksi_model.items():
    # Proses Training
    m_obj.fit(fitur_latih_w2v, y_train)

    # Proses Prediksi
    estimasi = m_obj.predict(fitur_uji_w2v)

    print(f"Strategi: {label}")
    print(classification_report(y_test, estimasi))
    print("-" * 30)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.9 MB/s eta 0:00:00

********** PERFORMA WORD2VEC **********

Strategi: SVM_Classifier
              precision    recall  f1-score   support

     Negatif       0.73      0.86      0.79        37
     Positif       0.76      0.57      0.65        28

    accuracy                           0.74        65
   macro avg       0.74      0.72      0.72        65
weighted avg       0.74      0.74      0.73        65

------------------------------
Strategi: Log_Regression
              precision    recall  f1-score   support

     Negatif       0.57      1.00      0.73        37
     Positif       0.00      0.00      0.00        28

    accuracy                           0.57        65
   macro avg       0.28      0.50      0.36        65
weighted avg       0.32      0.57      0.41        65

------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Percobaan 4 : GloVe Embedding**

In [29]:
# Unduh GloVe embeddings jika belum ada
!wget -nc http://nlp.stanford.edu/data/glove.6B.zip
!unzip -n glove.6B.zip

# --- 1. Memuat Pre-trained Embeddings GloVe ---
jalur_file_glove = "glove.6B.100d.txt"
kamus_glove = {}

with open(jalur_file_glove, "r", encoding="utf8") as berkas:
    for baris in berkas:
        komponen = baris.split()
        kata = komponen[0]
        koefisien = np.array(komponen[1:], dtype="float32")
        kamus_glove[kata] = koefisien


--2025-12-29 11:19:37--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-12-29 11:19:37--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-12-29 11:19:37--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [30]:
# --- 2. Fungsi Transformasi Kalimat ke Vektor ---
def transform_ke_glove(kalimat):
    daftar_kata = kalimat.split()
    # Mengumpulkan vektor kata jika tersedia di kamus GloVe
    koleksi_vektor = [kamus_glove[k] for k in daftar_kata if k in kamus_glove]

    # Mengambil nilai tengah (mean) dari seluruh vektor kata dalam satu kalimat
    if not koleksi_vektor:
        return np.zeros(100)
    return np.mean(koleksi_vektor, axis=0)

In [31]:
# --- 3. Pemrosesan Data Training dan Testing ---
# Mengonversi teks menjadi representasi numerik GloVe
fitur_glove_latih = np.stack(X_train.apply(transform_ke_glove))
fitur_glove_uji = np.stack(X_test.apply(transform_ke_glove))

In [33]:
# --- 4. Evaluasi Model ---
print(f"\n{'='*15} LAPORAN PERFORMA GLOVE {'='*15}\n")

for identitas, classifier in koleksi_model.items():
    # Pastikan menggunakan data GloVe (fitur_glove_latih), bukan W2V
    classifier.fit(fitur_glove_latih, y_train)

    prediksi_label = classifier.predict(fitur_glove_uji)

    print(f"Model: {identitas}")
    print(classification_report(y_test, prediksi_label))
    print("-" * 40)


=============== LAPORAN PERFORMA GLOVE ===============

Model: SVM_Classifier
              precision    recall  f1-score   support

     Negatif       0.75      0.81      0.78        37
     Positif       0.72      0.64      0.68        28

    accuracy                           0.74        65
   macro avg       0.73      0.73      0.73        65
weighted avg       0.74      0.74      0.74        65

----------------------------------------
Model: Log_Regression
              precision    recall  f1-score   support

     Negatif       0.77      0.73      0.75        37
     Positif       0.67      0.71      0.69        28

    accuracy                           0.72        65
   macro avg       0.72      0.72      0.72        65
weighted avg       0.73      0.72      0.72        65

----------------------------------------
